# Univariate Analysis

**Topic:** Exploratory Data Analysis

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Select** the correct chart type for a given column based on whether it is numeric or categorical
- **Interpret** a histogram and KDE curve to describe the shape, center, and spread of a distribution
- **Identify** skewness, modality, and extreme values from a single visualization

> **Tip:** Try `fare` with 20 bins, then drop the bins to 5. Notice how fewer bins hide the distribution shape. Then try `sex` or `pclass` and watch the widget automatically switch to a bar chart.

---
## How we got here

In `07_descriptive_statistics_in_practice.ipynb` we summarized each column with numbers. Now we move to pictures. Univariate analysis means examining one variable at a time, and visualization is the fastest way to reveal shape, outliers, and distribution patterns that numbers alone can miss. This builds directly on the visualization concepts from `statistics/08_data_visualization_statistics.ipynb`.

---
## Why this matters for data science

Before you look at any relationships between variables, you need to understand each variable individually. A feature that looks reasonable in a summary table might have a bimodal distribution, a spike at zero, or a long tail that will cause problems in a model. Univariate analysis is the step that catches those surprises. You cannot spot a bimodal distribution from a mean and standard deviation alone.

---
## Try it yourself

In [4]:
out = Output()

all_cols = [c for c in titanic.columns if c not in ["name", "ticket", "cabin", "boat", "body", "home.dest"]]
numeric_cols = titanic.select_dtypes(include="number").columns.tolist()
discrete_cols = [c for c in numeric_cols
                 if titanic[c].dtype.kind == "i" and titanic[c].nunique() <= 15]
continuous_cols = [c for c in numeric_cols if c not in discrete_cols]

col_dropdown = Dropdown(
    options=all_cols,
    value="age",
    description="Column:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

bin_slider = IntSlider(
    value=20, min=5, max=60, step=5,
    description="Bins:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="360px"),
)

def render(col, n_bins):
    series = titanic[col].dropna()
    bin_slider.disabled = col not in continuous_cols

    if col in continuous_cols:
        mean_v  = series.mean()
        median_v = series.median()
        std_v   = series.std()
        skew_v  = series.skew()

        kde_x = np.linspace(series.min(), series.max(), 400)
        kde_y = stats.gaussian_kde(series)(kde_x)
        kde_scale = len(series) * (series.max() - series.min()) / n_bins

        traces = [
            go.Histogram(
                x=series,
                nbinsx=n_bins,
                marker_color=PALETTE["primary"],
                opacity=0.7,
                name="Count",
            ),
            go.Scatter(
                x=kde_x,
                y=kde_y * kde_scale,
                mode="lines",
                line=dict(color=PALETTE["secondary"], width=2.5),
                name="KDE",
                yaxis="y",
            ),
            go.Scatter(
                x=[mean_v, mean_v], y=[0, kde_y.max() * kde_scale * 1.1],
                mode="lines",
                line=dict(color=PALETTE["accent"], width=2, dash="dash"),
                name=f"Mean ({mean_v:.1f})",
            ),
            go.Scatter(
                x=[median_v, median_v], y=[0, kde_y.max() * kde_scale * 1.1],
                mode="lines",
                line=dict(color=PALETTE["muted"], width=2, dash="dot"),
                name=f"Median ({median_v:.1f})",
            ),
        ]
        title = f"{col} — Histogram + KDE  (mean={mean_v:.1f}, median={median_v:.1f}, std={std_v:.1f}, skew={skew_v:.2f})"
        layout = base_layout(title=title, xaxis_title=col, yaxis_title="Count")
        layout.update(showlegend=True, height=380)

    elif col in discrete_cols:
        mean_v   = series.mean()
        median_v = series.median()
        vc = series.value_counts().sort_index()
        traces = [go.Bar(
            x=vc.index.astype(str).tolist(),
            y=vc.values,
            marker_color=PALETTE["primary"],
            text=vc.values,
            textposition="outside",
        )]
        title = f"{col} — Value Counts  (mean={mean_v:.2f}, median={median_v:.1f})"
        layout = base_layout(title=title, xaxis_title=col, yaxis_title="Count")
        layout.update(showlegend=False, height=380)

    else:
        vc = series.value_counts().sort_values(ascending=False)
        traces = [go.Bar(
            x=vc.index.astype(str).tolist(),
            y=vc.values,
            marker_color=PALETTE["primary"],
            text=vc.values,
            textposition="outside",
        )]
        title = f"{col} — Value Counts  (n={len(series)}, unique={series.nunique()})"
        layout = base_layout(title=title, xaxis_title=col, yaxis_title="Count")
        layout.update(showlegend=False, height=380)

    fig = go.Figure(data=traces, layout=layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def on_change(change):
    render(col_dropdown.value, bin_slider.value)

col_dropdown.observe(on_change, names="value")
bin_slider.observe(on_change, names="value")

display(VBox([HBox([col_dropdown, bin_slider]), out]))
render(col_dropdown.value, bin_slider.value)

---
## What's happening?

The right chart depends on the column type:

| Column type | Best chart | Why |
|-------------|-----------|-----|
| Numeric (continuous) | Histogram + KDE | Shows the full distribution shape, center, and tails |
| Numeric (discrete count) | Histogram or bar chart | Bin width matters — try different values |
| Categorical (few values) | Bar chart | Direct count comparison across categories |
| Categorical (many values) | Bar chart (top N) | Show only the most frequent categories |

The KDE (kernel density estimate) is a smoothed version of the histogram. It does not change the underlying data — it just makes the shape easier to read. When the KDE and the histogram disagree strongly, it usually means the bin count is too low.

---
## Real-world example: Titanic Dataset

The chart below shows the age distribution of Titanic passengers with a KDE overlay and lines marking the mean and median.

Notice:
- The distribution is **roughly bell-shaped** with a slight right skew — the mean (29.7) sits just above the median (28)
- There is a small **peak around age 1-2**, representing infants and toddlers who skew the lower tail
- The oldest passenger was 80, creating a short right tail

> **Discussion question:** The survival rate across the entire Titanic dataset is about 38%. Does that surprise you given what you know about the disaster? What does a 38% survival rate imply about who was saved?

In [3]:
np.random.seed(42)

age = titanic["age"].dropna()
kde_x = np.linspace(age.min(), age.max(), 400)
kde_y = stats.gaussian_kde(age)(kde_x)
kde_scale = len(age) * (age.max() - age.min()) / 25

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=age,
    nbinsx=25,
    marker_color=PALETTE["primary"],
    opacity=0.65,
    name="Passenger count",
))

fig.add_trace(go.Scatter(
    x=kde_x,
    y=kde_y * kde_scale,
    mode="lines",
    line=dict(color=PALETTE["secondary"], width=2.5),
    name="KDE (smoothed distribution)",
))

fig.add_vline(
    x=age.mean(), line_dash="dash", line_color=PALETTE["accent"],
    annotation_text=f"Mean: {age.mean():.1f}", annotation_position="top right",
)
fig.add_vline(
    x=age.median(), line_dash="dot", line_color=PALETTE["muted"],
    annotation_text=f"Median: {age.median():.1f}", annotation_position="top left",
)

layout = base_layout(
    title="Age Distribution of Titanic Passengers",
    xaxis_title="Age (years)",
    yaxis_title="Number of Passengers",
)
layout.update(showlegend=True)
fig.update_layout(layout)
fig.show()

### When to use each univariate chart

| Situation | Chart | Notes |
|-----------|-------|-------|
| Age, fare, salary (continuous) | Histogram + KDE | Bin count matters — try 15-30 for most datasets |
| Count of children, items ordered | Bar or histogram | Discrete values — integer bins |
| Gender, class, country (categorical) | Bar chart | Sort by frequency for readability |
| Time-based quantity (date, hour) | Line chart or histogram | Preserve time order in x-axis |
| Any column you want to summarize quickly | Box plot | Shows quartiles and outliers at a glance |

---
## Key takeaway

> **Always look at each variable individually before comparing them. The surprises you find in univariate analysis are often the most important things you will learn about your data.**

---
*Next up: 09_bivariate_analysis.ipynb — looking at relationships between pairs of variables, including the most important one: what predicts survival?*